# 01 · Tratamento de dados (Spark)

Pipeline em três camadas, cada uma salva em CSV dentro de `dados/`:

| Camada | Conteúdo | Regra |
|---|---|---|
| **bronze** | Abas do Excel como vieram, B01 e B02 empilhados | Nenhuma transformação, tudo como texto |
| **silver** | Tipos corrigidos, duplicados removidos, moeda imputada, câmbio corrigido, USD, classificação contábil, flags de fechamento | Uma linha por evento / lançamento |
| **gold** | Fatos agregados para análise, P&L mensal (visão oficial e atualizada), log de qualidade e conferências | Pronto para consumo |

In [1]:
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

import functions as fn

EXCEL_PATH = Path("OrionPay_Case_Senior.xlsx")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

spark = fn.get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/23 14:06:22 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/23 14:06:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/23 14:06:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Bronze

In [2]:
sheets = fn.read_excel_sheets(spark, EXCEL_PATH)

bronze = {
    "payment_events": fn.union_batches(sheets["B01_payment_events"], sheets["B02_payment_events"]),
    "financial_entries": fn.union_batches(sheets["B01_financial_entries"], sheets["B02_financial_entries"]),
    "merchants": sheets["merchants"],
    "countries": sheets["countries"],
    "account_mapping": sheets["account_mapping"],
    "fx_rates": sheets["fx_rates"],
    "close_calendar": sheets["close_calendar"],
}

for name, df in bronze.items():
    fn.write_csv(df, fn.BRONZE_DIR / f"{name}.csv")

pd.DataFrame({name: [df.count()] for name, df in bronze.items()}, index=["linhas"]).T

,linhas
payment_events,18548
financial_entries,89566
merchants,32
countries,4
account_mapping,11
fx_rates,100
close_calendar,25


## Silver

### Referências

O câmbio de **COP em mar/25 está 10× acima** dos demais meses (0,002462 contra ~0,00025). É um deslocamento de casa decimal: dividido por 10, o valor encaixa exatamente na sequência da série. A correção é feita por regra genérica (desvio > 3× da mediana da moeda, ajustado pela potência de 10 mais próxima) e a taxa original fica preservada para medir o impacto.

In [3]:
bronze = {name: fn.read_csv(spark, fn.BRONZE_DIR / f"{name}.csv") for name in bronze}

fx_rates = fn.correct_fx_outliers(fn.cast_fx_rates(bronze["fx_rates"]))
account_mapping = fn.cast_account_mapping(bronze["account_mapping"])
close_calendar = fn.cast_close_calendar(bronze["close_calendar"])
countries = bronze["countries"]
merchants = bronze["merchants"].withColumn("onboarded_at", F.to_date("onboarded_at"))

fn.to_pandas(fx_rates.filter("fx_is_corrected"))

,currency_code,period,usd_per_unit,fx_is_corrected,usd_per_unit_original
0,COP,2025-03-01,0.000246,True,0.002462


### Eventos de pagamento

In [4]:
payment_events = fn.cast_payment_events(bronze["payment_events"])
payment_events = fn.flag_tpv(payment_events)
payment_events = fn.convert_to_usd(payment_events, fx_rates, "amount_local")

fn.to_pandas(
    payment_events.groupBy("event_type", "status", "is_tpv")
    .agg(F.count("*").alias("eventos"), F.sum("amount_usd").alias("usd"))
    .orderBy("event_type", "status")
)

,event_type,status,is_tpv,eventos,usd
0,CAPTURE,CANCELLED,False,431,131031.94
1,CAPTURE,CONFIRMED,True,17130,5183194.81
2,CAPTURE,PENDING,False,639,195179.51
3,REFUND,CONFIRMED,False,348,26550.32


### Lançamentos financeiros

1. **Duplicados exatos**: 427 linhas repetidas (mesmo `entry_id` e todos os campos iguais), todas com competência jul/26. São removidas.
2. **Moeda vazia**: 170 lançamentos sem `currency_code`. Em 100% dos lançamentos com moeda, ela coincide com a moeda do país, então a moeda é imputada a partir do país.
3. **Classificação** pela regra vigente na `accounting_date`. A conta `Z900` não tem mapeamento: fica marcada como `UNMAPPED`, fora do Gross Profit e reportada à parte.
4. **Fechamento oficial**: `is_official` quando o mês está fechado e `posted_at ≤ official_cutoff_at`. `is_late_posting` marca o que chegou depois do cutoff.

In [5]:
financial_entries = fn.cast_financial_entries(bronze["financial_entries"])
rows_before_dedup = financial_entries.count()

financial_entries, removed_duplicates = fn.drop_exact_duplicates(financial_entries)
financial_entries = fn.impute_currency_from_country(financial_entries, countries)
financial_entries = fn.convert_to_usd(financial_entries, fx_rates, "amount")
financial_entries = fn.classify_entries(financial_entries, account_mapping)
financial_entries = fn.flag_official_close(financial_entries, close_calendar)
financial_entries = financial_entries.cache()

print(f"Antes: {rows_before_dedup:,} | depois: {financial_entries.count():,}")
fn.to_pandas(
    removed_duplicates.groupBy("period", "source_account")
    .agg(F.sum("removed_rows").alias("linhas_removidas"))
    .orderBy("source_account")
)

Antes: 89,566 | depois: 89,139


,period,source_account,linhas_removidas
0,2026-07-01,C100,85
1,2026-07-01,C110,85
2,2026-07-01,C120,85
3,2026-07-01,R100,85
4,2026-07-01,R110,85
5,2026-07-01,R190,2


In [6]:
fn.to_pandas(
    financial_entries.groupBy("source_account", "pl_group", "pl_subgroup", "pl_line")
    .agg(
        F.count("*").alias("lancamentos"),
        F.min("accounting_date").alias("de"),
        F.max("accounting_date").alias("ate"),
        F.sum("amount_usd").alias("usd"),
    )
    .orderBy("source_account", "de")
)

,source_account,pl_group,pl_subgroup,pl_line,lancamentos,de,ate,usd
0,C100,COST_OF_SALES,DIRECT_COSTS,Processing costs,17130,2024-09-01,2026-09-20,-64415.68
1,C110,COST_OF_SALES,DIRECT_COSTS,Network costs,17130,2024-09-01,2026-09-20,-17892.67
2,C120,COST_OF_SALES,DIRECT_COSTS,Platform costs,9974,2024-09-01,2025-12-31,-3641.61
3,C120,COST_OF_SALES,INDIRECT_COSTS,Platform costs,7156,2026-01-01,2026-09-20,-3515.86
4,C190,COST_OF_SALES,DIRECT_COSTS,Additional processing costs,1,2026-08-28,2026-08-28,-2400.00
5,C200,COST_OF_SALES,INDIRECT_COSTS,Shared service costs,100,2024-09-30,2026-09-01,-28008.00
6,C300,COST_OF_SALES,DIRECT_COSTS,FX settlement result,3033,2024-09-30,2026-09-20,-2092.86
7,R100,NET_REVENUE,PROCESSING_REVENUE,Processing fees,17130,2024-09-01,2026-09-20,154857.59
8,R110,NET_REVENUE,SERVICE_REVENUE,Service fees,17130,2024-09-01,2026-09-20,10366.45
9,R120,NET_REVENUE,REVENUE_ADJUSTMENTS,Contractual incentives,1,2026-05-20,2026-05-20,3000.00


### Validações da camada silver

In [7]:
validations = {
    "entry_id único": financial_entries.groupBy("entry_id").count().filter("count > 1").count() == 0,
    "event_id único": payment_events.groupBy("event_id").count().filter("count > 1").count() == 0,
    "lançamentos sem USD": financial_entries.filter(F.col("amount_usd").isNull()).count() == 0,
    "eventos sem USD": payment_events.filter(F.col("amount_usd").isNull()).count() == 0,
    "classificação não duplicou linhas": financial_entries.count() == financial_entries.select("entry_id").distinct().count(),
    "toda captura confirmada tem taxa R100": (
        payment_events.filter("is_tpv").select("payment_id")
        .subtract(financial_entries.filter("source_account = 'R100'").select("payment_id"))
        .count() == 0
    ),
}
pd.Series(validations, name="ok").to_frame()

,ok
entry_id único,True
event_id único,True
lançamentos sem USD,True
eventos sem USD,True
classificação não duplicou linhas,True
toda captura confirmada tem taxa R100,True


In [8]:
fn.write_csv(payment_events, fn.SILVER_DIR / "payment_events.csv")
fn.write_csv(financial_entries, fn.SILVER_DIR / "financial_entries.csv")
fn.write_csv(fx_rates, fn.SILVER_DIR / "fx_rates.csv")
fn.write_csv(account_mapping, fn.SILVER_DIR / "account_mapping.csv")
fn.write_csv(close_calendar, fn.SILVER_DIR / "close_calendar.csv")
fn.write_csv(merchants, fn.SILVER_DIR / "merchants.csv")
fn.write_csv(countries, fn.SILVER_DIR / "countries.csv");

## Gold

- `fact_tpv`: TPV por mês × país × merchant × produto × método × lote.
- `fact_pnl`: lançamentos agregados no mesmo grão + conta, linha da P&L e flags de fechamento. Valores sem merchant ficam como `UNATTRIBUTED`.
- `pnl_monthly`: P&L mensal nas visões **official** (somente o que estava registrado até o cutoff) e **updated** (tudo que chegou até B02).
- `dim_*`: cadastros para enriquecer os recortes.

In [9]:
fact_tpv = fn.build_fact_tpv(payment_events)
fact_pnl = fn.build_fact_pnl(financial_entries)

pnl_monthly = fn.monthly_pnl_spark(fact_pnl, fact_tpv, "official").unionByName(
    fn.monthly_pnl_spark(fact_pnl, fact_tpv, "updated")
)

fn.write_csv(fact_tpv, fn.GOLD_DIR / "fact_tpv.csv")
fn.write_csv(fact_pnl, fn.GOLD_DIR / "fact_pnl.csv")
fn.write_csv(pnl_monthly, fn.GOLD_DIR / "pnl_monthly.csv")
fn.write_csv(merchants, fn.GOLD_DIR / "dim_merchants.csv")
fn.write_csv(countries, fn.GOLD_DIR / "dim_countries.csv")
fn.write_csv(close_calendar, fn.GOLD_DIR / "dim_close_calendar.csv")

fn.to_pandas(
    pnl_monthly.filter("view = 'official'")
    .select("period", "tpv", "net_revenue", "cost_of_sales", "gross_profit", "take_rate", "gp_margin")
).round(4)

,period,tpv,net_revenue,cost_of_sales,gross_profit,take_rate,gp_margin
0,2024-09-01,125808.76,4549.08,-2492.62,2056.46,0.0362,0.4521
1,2024-10-01,135066.04,4791.58,-2684.66,2106.92,0.0355,0.4397
2,2024-11-01,170382.19,6017.27,-3254.91,2762.36,0.0353,0.4591
3,2024-12-01,171818.86,6091.28,-3350.00,2741.28,0.0355,0.4500
4,2025-01-01,152356.31,5276.89,-3130.32,2146.57,0.0346,0.4068
5,2025-02-01,155420.52,5409.02,-3246.31,2162.71,0.0348,0.3998
6,2025-03-01,166353.83,5666.37,-3499.05,2167.32,0.0341,0.3825
7,2025-04-01,174743.87,5967.61,-3685.83,2281.78,0.0342,0.3824
8,2025-05-01,182998.62,6176.07,-3894.50,2281.57,0.0337,0.3694
9,2025-06-01,184527.98,6241.30,-3983.93,2257.37,0.0338,0.3617


## Log de problemas de dados

Cada linha: o que é, quantos registros afeta, como foi tratado e o impacto em USD.

In [10]:
entries_closed = financial_entries.filter("period_status = 'CLOSED'")
fx_affected_entries = financial_entries.filter("fx_is_corrected")
fx_affected_events = payment_events.filter("fx_is_corrected AND is_tpv")
removed_usd = fn.convert_to_usd(
    fn.impute_currency_from_country(removed_duplicates, countries), fx_rates, "amount"
).withColumn("impact", F.col("amount_usd") * F.col("removed_rows"))
non_tpv = payment_events.filter("NOT is_tpv")
unattributed = financial_entries.filter("merchant_id IS NULL AND is_mapped")
c120_reclass = financial_entries.filter("source_account = 'C120' AND pl_subgroup = 'INDIRECT_COSTS'")
c300_favorable = financial_entries.filter("source_account = 'C300' AND amount > 0")
r120 = financial_entries.filter("source_account = 'R120'")
open_month = financial_entries.filter("period_status = 'OPEN'")


def usd_gap(df, original="amount_usd_original_fx", corrected="amount_usd"):
    return fn.sum_column(df, original) - fn.sum_column(df, corrected)


quality_log = pd.DataFrame(
    [
        {
            "id": "DQ01",
            "problema": "Lançamentos duplicados (entry_id e todos os campos idênticos)",
            "tabela": "financial_entries (B01)",
            "registros": int(fn.sum_column(removed_duplicates, "removed_rows")),
            "tratamento": "Removidos. Mantida uma ocorrência por entry_id",
            "impacto_usd": round(fn.sum_column(removed_usd, "impact"), 2),
            "efeito": "Sem tratamento, jul/26 teria receita e custos inflados (GP líquido indicado)",
        },
        {
            "id": "DQ02",
            "problema": "Câmbio COP de mar/25 com casa decimal deslocada (10×)",
            "tabela": "fx_rates",
            "registros": fx_affected_events.count() + fx_affected_entries.count(),
            "tratamento": "Taxa dividida por 10. Original preservada em usd_per_unit_original",
            "impacto_usd": round(usd_gap(fx_affected_events) , 2),
            "efeito": f"TPV de mar/25 superestimado no valor indicado; GP superestimado em {usd_gap(fx_affected_entries):,.2f}",
        },
        {
            "id": "DQ03",
            "problema": "Lançamentos sem currency_code",
            "tabela": "financial_entries",
            "registros": financial_entries.filter("currency_is_imputed").count(),
            "tratamento": "Moeda imputada pelo país (100% de aderência nos demais lançamentos)",
            "impacto_usd": round(fn.sum_column(financial_entries.filter("currency_is_imputed"), "amount_usd"), 2),
            "efeito": "Receita que seria perdida na conversão para USD",
        },
        {
            "id": "DQ04",
            "problema": "Conta Z900 sem regra no account_mapping",
            "tabela": "financial_entries",
            "registros": financial_entries.filter("NOT is_mapped").count(),
            "tratamento": "Fora do GP; reportada como UNMAPPED abaixo da P&L",
            "impacto_usd": round(fn.sum_column(financial_entries.filter("NOT is_mapped"), "amount_usd"), 2),
            "efeito": "Se for custo de venda, o GP de jun–ago/26 estaria superestimado no valor indicado",
        },
        {
            "id": "DQ05",
            "problema": "Lançamento postado após o cutoff de mês fechado",
            "tabela": "financial_entries (B02)",
            "registros": entries_closed.filter("is_late_posting").count(),
            "tratamento": "Fora da visão oficial; incluído na visão atualizada",
            "impacto_usd": round(fn.sum_column(entries_closed.filter("is_late_posting"), "amount_usd"), 2),
            "efeito": "GP atualizado de ago/26 menor que o oficial no valor indicado",
        },
        {
            "id": "DQ06",
            "problema": "Mudança de regra: C120 (Platform costs) passa de DIRECT para INDIRECT em jan/26",
            "tabela": "account_mapping",
            "registros": c120_reclass.count(),
            "tratamento": "Classificação conforme regra vigente; série pró-forma comparável na análise",
            "impacto_usd": round(fn.sum_column(c120_reclass, "amount_usd"), 2),
            "efeito": "Custos diretos caem a partir de jan/26 sem mudança econômica. GP não muda",
        },
        {
            "id": "DQ07",
            "problema": "Lançamentos sem merchant (C200, R120, C190)",
            "tabela": "financial_entries",
            "registros": unattributed.count(),
            "tratamento": "Mantidos na P&L como UNATTRIBUTED; fora do ranking de merchants",
            "impacto_usd": round(fn.sum_column(unattributed, "amount_usd"), 2),
            "efeito": "Diferença entre GP total e soma do GP por merchant",
        },
        {
            "id": "DQ08",
            "problema": "Eventos fora do TPV (REFUND, PENDING, CANCELLED)",
            "tabela": "payment_events",
            "registros": non_tpv.count(),
            "tratamento": "Excluídos do TPV conforme regra",
            "impacto_usd": round(fn.sum_column(non_tpv, "amount_usd"), 2),
            "efeito": "Volume que não entra no TPV",
        },
        {
            "id": "DQ09",
            "problema": "Evento pontual: incentivo contratual R120 (mai/26, MX, sem merchant)",
            "tabela": "financial_entries",
            "registros": r120.count(),
            "tratamento": "Mantido; destacado como não recorrente",
            "impacto_usd": round(fn.sum_column(r120, "amount_usd"), 2),
            "efeito": "Eleva o GP de mai/26",
        },
        {
            "id": "DQ10",
            "problema": "Resultado de FX favorável (C300 positivo) concentrado em mar/26",
            "tabela": "financial_entries",
            "registros": c300_favorable.count(),
            "tratamento": "Mantido (regra permite custo positivo); destacado como não recorrente",
            "impacto_usd": round(fn.sum_column(c300_favorable, "amount_usd"), 2),
            "efeito": "Eleva o GP de mar/26",
        },
        {
            "id": "DQ11",
            "problema": "Set/26 aberto e parcial (dados até 20/09)",
            "tabela": "payment_events / financial_entries",
            "registros": open_month.count(),
            "tratamento": "Sem visão oficial; usado só em leituras atualizadas e com ressalva",
            "impacto_usd": round(fn.sum_column(open_month.filter("is_mapped"), "amount_usd"), 2),
            "efeito": "GP parcial de set/26 (até 20/09). Não comparar com meses completos",
        },
    ]
)
quality_log.to_csv(fn.GOLD_DIR / "data_quality_log.csv", index=False)
quality_log

,id,problema,tabela,registros,tratamento,impacto_usd,efeito
0,DQ01,Lançamentos duplicados (entry_id e todos os ca...,financial_entries (B01),427,Removidos. Mantida uma ocorrência por entry_id,180.56,"Sem tratamento, jul/26 teria receita e custos ..."
1,DQ02,Câmbio COP de mar/25 com casa decimal deslocad...,fx_rates,672,Taxa dividida por 10. Original preservada em u...,250315.83,TPV de mar/25 superestimado no valor indicado;...
2,DQ03,Lançamentos sem currency_code,financial_entries,170,Moeda imputada pelo país (100% de aderência no...,1534.98,Receita que seria perdida na conversão para USD
3,DQ04,Conta Z900 sem regra no account_mapping,financial_entries,6,Fora do GP; reportada como UNMAPPED abaixo da P&L,-3600.00,"Se for custo de venda, o GP de jun–ago/26 esta..."
4,DQ05,Lançamento postado após o cutoff de mês fechado,financial_entries (B02),1,Fora da visão oficial; incluído na visão atual...,-2400.00,GP atualizado de ago/26 menor que o oficial no...
5,DQ06,Mudança de regra: C120 (Platform costs) passa ...,account_mapping,7156,Classificação conforme regra vigente; série pr...,-3515.86,Custos diretos caem a partir de jan/26 sem mud...
6,DQ07,"Lançamentos sem merchant (C200, R120, C190)",financial_entries,102,Mantidos na P&L como UNATTRIBUTED; fora do ran...,-27408.00,Diferença entre GP total e soma do GP por merc...
7,DQ08,"Eventos fora do TPV (REFUND, PENDING, CANCELLED)",payment_events,1418,Excluídos do TPV conforme regra,352761.77,Volume que não entra no TPV
8,DQ09,Evento pontual: incentivo contratual R120 (mai...,financial_entries,1,Mantido; destacado como não recorrente,3000.00,Eleva o GP de mai/26
9,DQ10,Resultado de FX favorável (C300 positivo) conc...,financial_entries,123,Mantido (regra permite custo positivo); destac...,374.04,Eleva o GP de mar/26


## Conferências

Totais preservados entre camadas, composição da P&L e concordância entre recortes.

In [11]:
silver_usd = fn.sum_column(financial_entries, "amount_usd")
gold_usd = fn.sum_column(fact_pnl, "amount_usd")
silver_tpv = fn.sum_column(payment_events.filter("is_tpv"), "amount_usd")
gold_tpv = fn.sum_column(fact_tpv, "tpv_usd")

official = fn.to_pandas(pnl_monthly.filter("view = 'official'"))
updated = fn.to_pandas(pnl_monthly.filter("view = 'updated'"))
closed = updated[updated["period"].isin(official["period"])]

gp_by_country = fn.to_pandas(fact_pnl.filter("is_official AND is_mapped").groupBy("country_code").agg(F.sum("amount_usd").alias("gp")))
gp_by_merchant = fn.to_pandas(fact_pnl.filter("is_official AND is_mapped").groupBy("merchant_id").agg(F.sum("amount_usd").alias("gp")))

official_b01 = fn.to_pandas(fn.monthly_pnl_spark(fact_pnl.filter("batch_id = 'B01'"), fact_tpv, "official"))

checks = pd.DataFrame(
    [
        ("Linhas bronze → silver (lançamentos)", rows_before_dedup, financial_entries.count() + fn.sum_column(removed_duplicates, "removed_rows")),
        ("Linhas bronze → silver (eventos)", bronze["payment_events"].count(), payment_events.count()),
        ("USD silver = USD fact_pnl", round(silver_usd, 2), round(gold_usd, 2)),
        ("TPV silver = TPV fact_tpv", round(silver_tpv, 2), round(gold_tpv, 2)),
        ("GP = Net Revenue + Cost of Sales (24 meses, oficial)", round(official["gross_profit"].sum(), 2), round((official["net_revenue"] + official["cost_of_sales"]).sum(), 2)),
        ("Updated − Official (meses fechados) = postagens tardias", round(closed["gross_profit"].sum() - official["gross_profit"].sum(), 2), round(fn.sum_column(entries_closed.filter("is_late_posting AND is_mapped"), "amount_usd"), 2)),
        ("Oficial B01 = Oficial B01+B02", round(official_b01["gross_profit"].sum(), 2), round(official["gross_profit"].sum(), 2)),
        ("Soma dos países = total (GP oficial)", round(gp_by_country["gp"].sum(), 2), round(official["gross_profit"].sum(), 2)),
        ("Soma dos merchants + não atribuído = total (GP oficial)", round(gp_by_merchant["gp"].sum(), 2), round(official["gross_profit"].sum(), 2)),
    ],
    columns=["conferência", "valor_a", "valor_b"],
)
checks["ok"] = (checks["valor_a"] - checks["valor_b"]).abs() < 0.01
checks.to_csv(fn.GOLD_DIR / "reconciliation_checks.csv", index=False)
checks

,conferência,valor_a,valor_b,ok
0,Linhas bronze → silver (lançamentos),89566.00,89566.00,True
1,Linhas bronze → silver (eventos),18548.00,18548.00,True
2,USD silver = USD fact_pnl,41860.16,41860.16,True
3,TPV silver = TPV fact_tpv,5183194.81,5183194.81,True
4,"GP = Net Revenue + Cost of Sales (24 meses, of...",47767.83,47767.83,True
5,Updated − Official (meses fechados) = postagen...,-2400.00,-2400.00,True
6,Oficial B01 = Oficial B01+B02,47767.83,47767.83,True
7,Soma dos países = total (GP oficial),47767.83,47767.83,True
8,Soma dos merchants + não atribuído = total (GP...,47767.83,47767.83,True


In [12]:
spark.stop()